In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import LabelEncoder

In [ ]:
# 데이터 불러오기
df = pd.read_csv('../data/무_월차낼게요.csv', encoding='cp949')
df_grow = pd.read_csv('../data/factor_external_weekly_ver_0721.csv', encoding='utf-8')

In [ ]:
df['총거래량(kg)'] = np.log1p(df['총거래량(kg)'])

In [ ]:
df.head()


In [ ]:
# year, week 파생
def format_week_int(weekno):
    year = weekno // 100
    week = weekno % 100
    return year, week

df_grow[['year', 'week']] = df_grow['weekno'].apply(lambda x: pd.Series(format_week_int(x)))
df_grow.drop(columns='weekno', inplace=True)

# 병합 전 품목코드 추가
df['item_code'] = 1101

# 외생변수 병합
df = pd.merge(
    df,
    df_grow[['year', 'week', 'item_code', 'holiday_flag', 'holiday_score', 'grow_score']],
    on=['year', 'week', 'item_code'],
    how='left'
)

# 병합 후 외생변수 컬럼 재정의
for col in ['holiday_flag', 'holiday_score', 'grow_score']:
    if f"{col}_y" in df.columns:
        df[col] = df[f"{col}_y"]
        df.drop(columns=[f"{col}_x", f"{col}_y"], inplace=True)
df.drop(columns='item_code', inplace=True)

In [ ]:
df.columns

In [ ]:
# 불필요 컬럼 제거
# df = df.drop(columns=['등급코드', '일평균기온', '최고기온', '최저기온', '평균상대습도', '강수량(mm)', '1시간최고강수량(mm)'], errors='ignore')

In [ ]:
# 0. 최소 연도 확인 (예: 2018)
min_year = df['year'].min()

# 1. 병합 대상 테이블 생성 (중복 제거 중요!)
price_lag = (
    df[['year', 'week', '직팜산지코드', '품종코드', '평균단가(원)']]
    .drop_duplicates(subset=['year', 'week', '직팜산지코드', '품종코드'])
    .copy()
)
price_lag['year'] += 1
price_lag = price_lag.rename(columns={'평균단가(원)': 'price_lag_1y'})

# 2. 병합 수행 (1:1 보장)
df = pd.merge(
    df,
    price_lag,
    on=['year', 'week', '직팜산지코드', '품종코드'],
    how='left'
)

# 3. 월 기준 평균단가로 보완
df['month'] = pd.to_datetime(df['week_start']).dt.month

monthly_avg = (
    df[['year', 'month', '직팜산지코드', '품종코드', '평균단가(원)']]
    .groupby(['year', 'month', '직팜산지코드', '품종코드'], as_index=False)
    .mean()
)

monthly_avg['year'] += 1
monthly_avg = monthly_avg.rename(columns={'평균단가(원)': 'price_lag_month_fill'})

df = pd.merge(
    df,
    monthly_avg,
    on=['year', 'month', '직팜산지코드', '품종코드'],
    how='left'
)

# 📌 월평균으로 보완 (단, 최소 연도는 제외)
df['price_lag_1y'] = df.apply(
    lambda row: row['price_lag_month_fill']
    if pd.isna(row['price_lag_1y']) and row['year'] > min_year
    else row['price_lag_1y'],
    axis=1
)

# 4. 연 평균으로 fallback
yearly_avg = (
    df[['year', '직팜산지코드', '품종코드', '평균단가(원)']]
    .groupby(['year', '직팜산지코드', '품종코드'], as_index=False)
    .mean()
)
yearly_avg['year'] += 1
yearly_avg = yearly_avg.rename(columns={'평균단가(원)': 'price_lag_fallback'})

df = pd.merge(
    df,
    yearly_avg,
    on=['year', '직팜산지코드', '품종코드'],
    how='left'
)

df['price_lag_1y'] = df.apply(
    lambda row: row['price_lag_fallback']
    if pd.isna(row['price_lag_1y']) and row['year'] > min_year
    else row['price_lag_1y'],
    axis=1
)
df.dropna(subset=['price_lag_1y'], inplace=True)

In [ ]:
print("병합 후 행 수:", len(df))
print("NaN 비율:", df['price_lag_1y'].isna().mean())

In [ ]:
# df.drop(columns=['month', 'price_lag_month_fill', 'price_lag_fallback'], inplace=True, errors='ignore')

df.sample(5)

In [ ]:
print("남은 컬럼:\n", df.columns.tolist())


In [ ]:
# y, X 분리
y = df['평균단가(원)']
X = df.drop(columns=['평균단가(원)'])

In [ ]:
# 이동평균 및 EMA 파생 변수 생성
# SMA : 단순히 평균한 것
# EMA : 지수이동평균, 최근 데이터에 더 많은 가중치를 부여

def generate_moving_averages(df, target_col='평균단가(원)', group_cols=['직팜산지코드', '품종코드'], windows=[4, 13, 26]):
    df = df.sort_values(group_cols + ['year', 'week']).copy()
    for w in windows:
        df[f'SMA_{w}'] = df.groupby(group_cols)[target_col].transform(lambda x: x.rolling(window=w, min_periods=1).mean())
        df[f'EMA_{w}'] = df.groupby(group_cols)[target_col].transform(lambda x: x.ewm(span=w, adjust=False).mean())
    df['EMA4_SMA4_diff'] = df['EMA_4'] - df['SMA_4']
    df['EMA13_SMA13_diff'] = df['EMA_13'] - df['SMA_13']
    df['EMA26_SMA26_diff'] = df['EMA_26'] - df['SMA_26']
    df['EMA_4_rate'] = df.groupby(group_cols)['EMA_4'].transform(lambda x: x.pct_change().fillna(0))
    df['EMA_13_rate'] = df.groupby(group_cols)['EMA_13'].transform(lambda x: x.pct_change().fillna(0))
    df['EMA_26_rate'] = df.groupby(group_cols)['EMA_26'].transform(lambda x: x.pct_change().fillna(0))
    return df

df = generate_moving_averages(df)

In [ ]:
moving_avg_cols = [
    'SMA_4', 'SMA_13', 'SMA_26',
    'EMA_4', 'EMA_13', 'EMA_26',
    'EMA4_SMA4_diff', 'EMA13_SMA13_diff', 'EMA26_SMA26_diff',
    'EMA_4_rate', 'EMA_13_rate', 'EMA_26_rate', 'holiday_flag',
    'holiday_score', 'grow_score', 'price_lag_1y'
]

df = df.dropna(subset=moving_avg_cols).copy()

# # 이동평균 변수 스케일링
# scaler = RobustScaler()
# X_scaled = scaler.fit_transform(df[moving_avg_cols])
# df_scaled = pd.DataFrame(X_scaled, columns=moving_avg_cols, index=df.index)
# df.update(df_scaled)

In [ ]:
# 모델 학습용 재분리 및 주기형 변수 처리
y = df['평균단가(원)']
X = df.drop(columns=['평균단가(원)'])

# ✅ 여기서 제거하는 게 베스트!
X = X.drop(columns=[
    '등급코드', '일평균기온', '최고기온', '최저기온',
    '평균상대습도', '강수량(mm)', '1시간최고강수량(mm)',
    'month', 'price_lag_month_fill', 'price_lag_fallback'
], errors='ignore')

X['week_start'] = pd.to_datetime(X['week_start'])
X['year'] = X['week_start'].dt.year
X['week'] = X['week_start'].dt.isocalendar().week.astype(int)

X['week_sin'] = np.sin(2 * np.pi * X['week'] / 52)
X['week_cos'] = np.cos(2 * np.pi * X['week'] / 52)
X = X.drop(columns=['week'])


In [ ]:
X_train = X[X['year'] < 2025].drop(columns=['week_start'])
y_train = y[X['year'] < 2025]
X_test = X[X['year'] == 2025].drop(columns=['week_start'])
y_test = y[X['year'] == 2025]


In [ ]:
from sklearn.preprocessing import RobustScaler

exclude_cols = ['year', '품종코드', '직팜산지코드']  # 범주형 및 예외
numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns
scale_cols = [col for col in numerical_cols if col not in exclude_cols]

scaler = RobustScaler()
X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test[scale_cols] = scaler.transform(X_test[scale_cols])

In [ ]:
cat_cols = ['직팜산지코드', '품종코드']

for col in cat_cols:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col])
    X_test[col] = le.transform(X_test[col])

In [ ]:
# df.sample(5)
X_train.isna().sum(), X_test.isna().sum()
# train_df.info()
# test_df.info()

In [ ]:
param_distributions = {
    'n_estimators': [100, 300],            # 트리 개수: 충분한 범위
    'max_depth': [10, None],               # 제한 or 무제한
    'min_samples_split': [2, 5],           # 노드 분할 최소 샘플 수
    'min_samples_leaf': [1, 2],            # 리프 노드 최소 샘플 수
    'max_features': ['sqrt'],              # 가장 일반적인 설정
}


search = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=42, n_jobs=-1),
    param_distributions=param_distributions,
    n_iter=5,
    cv=3,
    verbose=1,
    n_jobs=-1,
    scoring='neg_root_mean_squared_error'
)

search.fit(X_train, y_train)
model = search.best_estimator_
pred = model.predict(X_test)


In [ ]:
# ✅ 성능 지표 계산
rmse = np.sqrt(mean_squared_error(y_test, pred))
mae = mean_absolute_error(y_test, pred)
mape = mean_absolute_percentage_error(y_test, pred) * 100  # %

r2 = r2_score(y_test, pred)

# ✅ 출력
print("\n🏁 모델 성능:")
print(f"📉 RMSE: {rmse:.2f}")
print(f"📊 MAE : {mae:.2f}")
print(f"📊 MAPE: {mape:.2f}%")
print(f"📈 R²   : {r2:.4f}")

In [ ]:
model

In [ ]:
# 중요도 추출
importances = model.feature_importances_
feature_names = X_train.columns

# Series로 보기 좋게
import pandas as pd
feat_importance = pd.Series(importances, index=feature_names).sort_values(ascending=False)

# 출력
print("\n🔍 변수 중요도 Top 10")
print(feat_importance.head(10))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
plt.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
feat_importance.head(20).plot(kind='bar')
plt.title('📊 Feature Importance (Top 20)')
plt.ylabel('Importance Score')
plt.tight_layout()
plt.show()


In [ ]:
# 1. 예측 결과와 실제값 결합
df_result = pd.DataFrame({
    'index': y_test.index,
    '실제단가': y_test.values,
    '예측단가': pred
})

# 2. 원본 df에서 'year', 'week' 컬럼만 추출해서 join
df_meta = df[['year', 'week']].reset_index()  # 원본 df에서 날짜 정보 추출
df_result = df_result.merge(df_meta, how='left', on='index')  # index 기준으로 날짜 정보 병합

# 3. 주차별 평균 계산
df_weekly = df_result.groupby(['year', 'week'])[['실제단가', '예측단가']].mean().reset_index()
# 1. year와 week를 문자열로 붙인 새 컬럼 생성 (예: '2025-01')
df_weekly['연도주차'] = df_weekly['year'].astype(str) + '-' + df_weekly['week'].astype(str).str.zfill(2)


# 4. 시각화
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 6))
plt.plot(df_weekly['연도주차'], df_weekly['실제단가'], label='실제단가', marker='o')
plt.plot(df_weekly['연도주차'], df_weekly['예측단가'], label='예측단가', marker='x')
plt.title('주차별 평균단가 비교')
plt.xlabel('연도-주차')
plt.ylabel('단가(원)')
plt.xticks(rotation=45)  # x축 라벨이 겹치지 않도록 회전
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.inspection import permutation_importance

result = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=42, scoring='neg_root_mean_squared_error')
perm_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance": result.importances_mean
}).sort_values("importance", ascending=False)
